In [1]:
import pandas as pd
import os

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [48]:
data_path = './input/OSO_CTD_O35_2025-06-18_09_04_19_raw.csv'
instrument_name = 'OSO_CTD_O35' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"
csv_encoding = 'utf-8' # or 'latin-1' depending on your file encoding

## Étape 1 : Charger les données CSV

In [49]:
df = pd.read_csv(
    data_path,
    low_memory=False,
    delimiter=';',
    encoding=csv_encoding
)

# Supprime les colonnes entièrement vides
df = df.dropna(axis=1, how='all')

print(df.head())

          Date & heure  Profondeur(m)  Temperature mer (C)
0  2025-06-18 09:04:19           0.16                25.16
1  2025-06-18 09:34:09           0.18                24.72
2  2025-06-18 10:04:02           0.19                24.32
3  2025-06-18 10:33:54           0.19                24.04
4  2025-06-18 11:03:46           0.19                23.47


## Étape 2 : Adapter au format de la plateforme de donnée

In [50]:

if {'Date & heure'}.issubset(df.columns):
    df['datetime'] = pd.to_datetime(
        df['Date & heure'].astype(str).str.strip(),
        errors='coerce',
        format='%Y-%m-%d %H:%M:%S'
    )
    df = df.drop(columns=['Date & heure'], errors='ignore')

# Renommer les colonnes pour correspondre au format de la plateforme
df = df.rename(columns={
    'Profondeur(m)': 'sea_depth',
    'Temperature mer (C)': 'sea_temp',
    'Conductivity(ms/cm)': 'ec_abs'
})

print(df.head())

   sea_depth  sea_temp            datetime
0       0.16     25.16 2025-06-18 09:04:19
1       0.18     24.72 2025-06-18 09:34:09
2       0.19     24.32 2025-06-18 10:04:02
3       0.19     24.04 2025-06-18 10:33:54
4       0.19     23.47 2025-06-18 11:03:46


## Étape 3 : Exporter le résultat

In [51]:
os.makedirs(output_dir, exist_ok=True)

if not df.empty:
    start_at = df['datetime'].min().strftime('%Y-%m-%d_%H_%M_%S')
    output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}.csv")
    df.to_csv(output_csv_final, sep=';', date_format='%Y-%m-%dT%H:%M:%SZ', index=False)
    print(f"Fichier exporté : {output_csv_final}")
else:
    print("Aucune donnée interpolée à exporter.")

Fichier exporté : output/OSO_CTD_O35_2025-06-18_09_04_19.csv
